# Notebook 5 — Adaptive Escalation Framework (End-to-End)
Loads LSTM · Isolation Forest · Scaler → runs full pipeline → generates all plots.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd

from modules.predictor        import load_lstm, predict_horizon
from modules.anomaly_detector import load_isolation_forest, batch_scores, batch_labels
from modules.escalation       import run_escalation
from modules.resource_allocator import allocate_resources
from modules.optimizer        import compare_strategies, compute_cost, energy_model
from modules.dashboard        import (
    fig_traffic_prediction, fig_anomaly_scores, fig_confidence_gauge,
    fig_llm_invocations, fig_qos_dashboard, fig_cost_comparison, fig_slice_allocation
)
from modules.preprocessing    import FEATURE_COLS
from config import DATA_PATH, THRESHOLD_PATH, DEFAULT_THRESHOLD, SEQ_LEN

print('All modules loaded.')

## 1. Load Models & Threshold

In [ ]:
lstm_model = load_lstm()
iso_clf    = load_isolation_forest()

try:
    with open(THRESHOLD_PATH) as f:
        tau = json.load(f)['optimal_threshold']
except FileNotFoundError:
    tau = DEFAULT_THRESHOLD
print(f'Threshold τ = {tau:.4f}')

## 2. Load Dataset & Run Pipeline on Test Window

In [ ]:
df = pd.read_csv(DATA_PATH)
numeric_feat_cols = [c for c in FEATURE_COLS if c in df.columns]
X_feat = df[numeric_feat_cols].values
traffic_series = df['traffic_mbps'].values

# Use last 200 samples as demo window
DEMO_START = len(df) - 200
results = []

for i in range(DEMO_START, len(df)):
    window = traffic_series[max(0, i - SEQ_LEN): i]
    if len(window) < SEQ_LEN:
        window = np.pad(window, (SEQ_LEN - len(window), 0))

    pred_traffic = float(lstm_model.predict(
        window.reshape(1, SEQ_LEN, 1), verbose=0)[0, 0])

    a_score = float(batch_scores(iso_clf, X_feat[i:i+1])[0])

    row = df.iloc[i]
    res = run_escalation(
        actual_traffic   = float(row['traffic_mbps']),
        predicted_traffic= pred_traffic,
        anomaly_score    = a_score,
        latency_ms       = float(row.get('latency_ms', 5)),
        bandwidth_mbps   = float(row.get('bandwidth_mbps', 80)),
        reliability      = float(row.get('reliability', 99.9)),
        device_type      = str(row.get('device_type', 'ICU Monitoring')),
        threshold        = tau,
        emergency        = False,
    )
    alloc = allocate_resources(
        pred_traffic, res['slice'], res['llm_invoked'], str(row.get('device_type','ICU Monitoring'))
    )
    res.update(alloc)
    results.append(res)

print(f'Pipeline complete. LLM invocations: {sum(r["llm_invoked"] for r in results)}/{len(results)}')

## 3. Plots

In [ ]:
timestamps  = list(range(len(results)))
actual_arr  = np.array([r['actual_traffic']    for r in results])
pred_arr    = np.array([r['predicted_traffic'] for r in results])
a_scores    = np.array([r['anomaly_score']     for r in results])
a_labels    = np.array([1 if r['anomaly_score'] > 0.5 else 0 for r in results])
conf_arr    = np.array([r['confidence']        for r in results])
esc_arr     = np.array([r['escalation_score']  for r in results])
qos_arr     = np.array([r['qos_score']         for r in results])
bw_arr      = np.array([r['allocated_bandwidth'] for r in results])
lat_arr     = np.array([r['allocated_latency']   for r in results])
llm_flags   = [r['llm_invoked'] for r in results]

fig_traffic_prediction(timestamps, actual_arr, pred_arr).show()
fig_anomaly_scores(timestamps, a_scores, a_labels).show()
fig_confidence_gauge(float(conf_arr.mean()), float(esc_arr.mean()), tau).show()
fig_llm_invocations(timestamps, llm_flags).show()
fig_qos_dashboard(timestamps, qos_arr, bw_arr, lat_arr).show()

## 4. Strategy Comparison & Cost Function

In [ ]:
comparison = compare_strategies(results)
fig_cost_comparison(comparison).show()

print('\n=== Strategy Comparison ===')
for strategy, metrics in comparison.items():
    print(f'\n{strategy}:')
    for k, v in metrics.items():
        print(f'  {k:25s}: {v:.4f}')

## 5. Last Slice Allocation

In [ ]:
last_slice = results[-1]['slice']
fig_slice_allocation(last_slice).show()
print('Last slice recommendation:')
print(json.dumps(last_slice, indent=2))
print('\nNotebook 5 complete.')